In [19]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA 

import os 
from langchain_core.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder
)
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
llm = ChatNVIDIA(
    model="meta/llama-3.1-70b-instruct",
    api_key=os.getenv("NVIDIA_API_KEY"),
)

In [4]:
template = ChatPromptTemplate.from_template("{prompt}")

chain = template | llm | StrOutputParser()
about = "My name is abhilov Gupta, I am a software engineer with a passion for learning and problem-solving. I have experience in various programming languages and frameworks, and I enjoy working on projects that challenge me to think creatively. In my free time, I like to explore new technologies and contribute to open-source projects."

chain.invoke({"prompt": about})

"It's great to meet you, Abhilov Gupta.  As a software engineer, you have a solid foundation in various programming languages and frameworks, which is excellent for tackling complex projects. Your passion for learning and problem-solving is also a valuable asset, as it allows you to stay adaptable and innovative in the ever-evolving tech landscape.\n\nIt's also wonderful that you enjoy exploring new technologies and contributing to open-source projects in your free time. This not only helps you stay up-to-date with the latest developments but also gives you the opportunity to collaborate with other developers, share knowledge, and give back to the community.\n\nWhat specific areas of software engineering interest you the most, and what kind of projects do you usually enjoy working on? Are there any particular technologies or frameworks that you're currently exploring or would like to learn more about?"

In [5]:
prompt2 = "what is my name?"
chain.invoke({"prompt": prompt2})

"I don't have any information about your name. Our conversation just started, and I don't retain any personal data about users. If you'd like to share your name, I'd be happy to chat with you."

#### Till now it do not have any history of mine, let's start with the message history part to store the chat history.

## Runnable with message History

In [6]:
from langchain_core.messages import HumanMessage
from langchain_core.runnables.history import RunnableWithMessageHistory #this will be used to create a custom chain that can maintain a history of messages
from langchain_community.chat_message_histories import SQLChatMessageHistory

In [7]:
def get_session_history(session_id):
    return SQLChatMessageHistory(session_id=session_id, connection_string="sqlite:///chat_history.db")

In [8]:
runnable_with_history = RunnableWithMessageHistory(chain, get_session_history=get_session_history)

In [9]:
user_id = "Abhilov_Gupta"
history = get_session_history(user_id)
history.add_user_message("My name is Abhilov Gupta, I am a software engineer with a passion for learning and problem-solving. I have experience in various programming languages and frameworks, and I enjoy working on projects that challenge me to think creatively. In my free time, I like to explore new technologies and contribute to open-source projects.")

C:\Users\Abhilov\AppData\Local\Temp\ipykernel_25272\2701367014.py:2: LangChainDeprecationWarning: `connection_string` was deprecated in LangChain 0.2.2 and will be removed in 1.0. Use connection instead.
  history = get_session_history(user_id)


In [15]:
history.get_messages()

[HumanMessage(content='My name is abhilov Gupta, I am a software engineer with a passion for learning and problem-solving. I have experience in various programming languages and frameworks, and I enjoy working on projects that challenge me to think creatively. In my free time, I like to explore new technologies and contribute to open-source projects.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='It seems like you\'ve provided a structured representation of a message in a format that\'s commonly used in natural language processing or chatbot applications. \n\nHere\'s a breakdown of the message:\n\n- `content`: This is the main text of the message. It appears to be a personal introduction from someone named Abhilov Gupta, who is a software engineer with a passion for learning and problem-solving.\n\n- `additional_kwargs`: This is an empty dictionary, suggesting that there are no additional keyword arguments provided with this message.\n\n- `response_metadata`: This i

In [11]:
history.clear() # this will clear the history of messages for the user_id "Abhilov_Gupta"

In [12]:
runnable_with_history.invoke(
    [HumanMessage(content=about)],
    config={"configurable": {"session_id": user_id}},
)

'It seems like you\'ve provided a structured representation of a message in a format that\'s commonly used in natural language processing or chatbot applications. \n\nHere\'s a breakdown of the message:\n\n- `content`: This is the main text of the message. It appears to be a personal introduction from someone named Abhilov Gupta, who is a software engineer with a passion for learning and problem-solving.\n\n- `additional_kwargs`: This is an empty dictionary, suggesting that there are no additional keyword arguments provided with this message.\n\n- `response_metadata`: This is also an empty dictionary, indicating that there is no metadata associated with this message.\n\nIf you were to write a response to this message, you could say something like:\n\n"Hello Abhilov, it\'s great to hear about your passion for learning and problem-solving. What kind of projects have you been working on lately, and are there any new technologies that you\'re particularly excited about?"'

In [13]:
question = "what is my name?"

runnable_with_history.invoke(
    [HumanMessage(content=question)],
    config={"configurable": {"session_id": user_id}},
)

'Your name is Abhilov Gupta.'

In [16]:
runnable_with_history.invoke(
    [HumanMessage(content="tell me about myself")],
    config={"configurable": {"session_id": user_id}},
)

'Based on the conversation, I can infer that you are Abhilov Gupta, a software engineer with a passion for learning and problem-solving. You have experience in various programming languages and frameworks, and you enjoy working on projects that challenge you to think creatively. In your free time, you like to explore new technologies and contribute to open-source projects.\n\nHere\'s a possible response:\n\n"It seems like you\'re looking for a summary of what you\'ve shared about yourself. To recap, you\'re a software engineer with a passion for learning and problem-solving. You have experience in various programming languages and frameworks, and you enjoy working on projects that challenge you to think creatively. You also like to explore new technologies and contribute to open-source projects in your free time. Is there anything specific you\'d like to know or talk about regarding your interests or experiences?"'

# Message History with Directory Like Inputs

In [20]:
system = SystemMessagePromptTemplate.from_template("You are a helpful assistant that remembers the user's information and can answer questions about the user based on the information provided by the user in the past.")

human = HumanMessagePromptTemplate.from_template("{prompt}")

# we can create a prompt template that includes the system message, the history of messages, and the human message. The MessagesPlaceholder will be replaced with the history of messages for the user_id "Abhilov_Gupta" when the chain is invoked.
messages = [system, MessagesPlaceholder(variable_name="history"), human]

prompt =ChatPromptTemplate(messages=messages)

chain = prompt | llm | StrOutputParser()

runnable_with_history = RunnableWithMessageHistory(chain, get_session_history=get_session_history, input_messages_key='input', history_messages_key='history')

In [21]:
def chat_with_llm(session_id, input):
    output = runnable_with_history.invoke({'input':input}, config={"configurable": {"session_id": session_id}})
    return output

In [25]:
chat_with_llm(user_id, about)

"Nice to meet you, Abhilov Gupta. I've taken note of the information you provided. To recap, I now know that:\n\n1. Your name is Abhilov Gupta.\n2. You are a software engineer.\n3. You have a passion for learning and problem-solving.\n4. You have experience in various programming languages and frameworks.\n5. You enjoy working on challenging projects that require creative thinking.\n6. In your free time, you like to explore new technologies and contribute to open-source projects.\n\nFeel free to share more about yourself, and I'll do my best to remember it for our conversation."

In [26]:
chat_with_llm(user_id, "what is my name?")

'Your name is Abhilov Gupta.'

In [27]:
chat_with_llm(user_id, "what i like to do in my free time?")

'In your free time, you like to explore new technologies and contribute to open-source projects.'